# Database Export

In [8]:
from pyspark.sql import SparkSession
import sqlite3
import pandas as pd

spark = (
    SparkSession.builder
    .appName("BusServiceReliability")
    .master("local[*]")
    .getOrCreate()
)

df = spark.read.parquet("../outputs/cleaned_timetable_parquet")
df.show(5, truncate=False)

+-------------------------------------------------------------------------------------------------------+----------+----------+------------+-------------+----------+
|filename                                                                                               |journey_id|operator  |service_code|stop_sequence|route_size|
+-------------------------------------------------------------------------------------------------------+----------+----------+------------+-------------+----------+
|10A-None--SCMY-GM-2026-07-26-Gillmoss_July_2026_EV_ADDED_FINAL__SCMY_PC1033334_5_20260719-BODS_V1_1.xml|VJ2904    |Stagecoach|PC1033334:5 |55           |Long      |
|10A-None--SCMY-GM-2026-07-26-Gillmoss_July_2026_EV_ADDED_FINAL__SCMY_PC1033334_5_20260719-BODS_V1_1.xml|VJ2904    |Stagecoach|PC1033334:5 |56           |Long      |
|10A-None--SCMY-GM-2026-07-26-Gillmoss_July_2026_EV_ADDED_FINAL__SCMY_PC1033334_5_20260719-BODS_V1_1.xml|VJ2904    |Stagecoach|PC1033334:5 |57           |Long      |
|10A

## Build Service-Level Summary Table

In [9]:
from pyspark.sql.functions import countDistinct, stddev, max as spark_max, first

service_summary = df.groupBy("service_code", "operator", "route_size").agg(
    countDistinct("journey_id").alias("num_journeys"),
    spark_max("stop_sequence").alias("max_stop_sequence"),
    stddev("stop_sequence").alias("stop_seq_std")
)

service_summary_pd = service_summary.toPandas()
print("Rows:", len(service_summary_pd))
service_summary_pd.head()

Rows: 244


,service_code,operator,route_size,num_journeys,max_stop_sequence,stop_seq_std
0,PC1033334:15,Stagecoach,Long,148,62,3.492184
1,PC1033334:596,Stagecoach,Short,2,14,3.953901
2,PC1033334:14,Stagecoach,Medium,336,45,5.279301
3,PC1033334:15,Stagecoach,Medium,551,49,8.254396
4,PC1033334:594,Stagecoach,Medium,3,37,4.703992


## Write to SQLite Database

In [10]:
db_path = "../outputs/bus_reliability.db"
conn = sqlite3.connect(db_path)

service_summary_pd.to_sql("service_summary", conn, if_exists="replace", index=False)

# Also store the model comparison results table for reference
model_results = pd.DataFrame([
    {"model": "Logistic Regression", "accuracy": 0.8576, "precision": 0.8224, "recall": 0.8576, "f1_score": 0.8303, "roc_auc": 0.8464},
    {"model": "Random Forest", "accuracy": 0.8624, "precision": 0.7438, "recall": 0.8624, "f1_score": 0.7987, "roc_auc": 0.5000},
    {"model": "Decision Tree", "accuracy": 0.8511, "precision": 0.7438, "recall": 0.8511, "f1_score": 0.7932, "roc_auc": 0.4937},
])
model_results.to_sql("model_results", conn, if_exists="replace", index=False)

conn.commit()
print("Database created at", db_path)

Database created at ../outputs/bus_reliability.db


## Parameterized Queries (SQL Injection Prevention)

In [11]:
# Parameterized query example — using ? placeholders instead of string concatenation
# This is the safe pattern that prevents SQL injection

def get_routes_by_size(conn, route_size):
    """Safely query routes by size using a parameterized query."""
    query = "SELECT service_code, operator, num_journeys, max_stop_sequence FROM service_summary WHERE route_size = ?"
    return pd.read_sql_query(query, conn, params=(route_size,))

# Example 1: Get all Long routes
long_routes = get_routes_by_size(conn, "Long")
print("Long routes:", len(long_routes))
long_routes.head()

Long routes: 25


,service_code,operator,num_journeys,max_stop_sequence
0,PC1033334:15,Stagecoach,148,62
1,PC1033334:578,Stagecoach,9,60
2,PC1033334:375,Stagecoach,1,51
3,PC1033334:186,Stagecoach,236,88
4,PC1033334:593,Stagecoach,2,71


In [13]:
def get_model_performance(conn, model_name):
    """Safely query a specific model's performance using a parameter."""
    query = "SELECT * FROM model_results WHERE model = ?"
    return pd.read_sql_query(query, conn, params=(model_name,))

get_model_performance(conn, "Logistic Regression")

,model,accuracy,precision,recall,f1_score,roc_auc
0,Logistic Regression,0.8576,0.8224,0.8576,0.8303,0.8464


## Export SQL Dump

In [14]:
with open("../outputs/bus_reliability_dump.sql", "w") as f:
    for line in conn.iterdump():
        f.write(f"{line}\n")

print("SQL dump saved to ../outputs/bus_reliability_dump.sql")

SQL dump saved to ../outputs/bus_reliability_dump.sql


## Database Storage & Security Summary

A SQLite database (`bus_reliability.db`) was created to satisfy the submission's Database
Export requirement, containing two tables: `service_summary` (244 rows — one per unique
service_code, aggregated from the 271,552-row raw dataset) and `model_results` (the 3-model
comparison from the evaluation notebook).

**SQL injection prevention**: All queries against this database use parameterized queries
via Python's DB-API `?` placeholder syntax (e.g. `WHERE route_size = ?` with `params=(value,)`),
rather than string concatenation or f-string interpolation. This ensures user-supplied values
are always treated as data, never as executable SQL — the standard defence against SQL
injection attacks.

**Relationship to the main pipeline**: this SQLite database supplements, rather than
replaces, the Parquet-based storage used throughout the main pipeline (justified in
`01_data_ingestion.ipynb`). Parquet remains the primary store for the full 271K-row dataset
given its columnar/distributed advantages at scale; SQLite here demonstrates the required
relational storage pattern and parameterized query security practice on a smaller, purpose-built
summary table.